In [0]:
from pyspark.sql.functions import col, avg, count, stddev, percentile_approx

In [0]:
# Read from silver table
df_silver = spark.table("workspace.taxi_schema.taxi_silver")

# Calculate price per mile for trips with distance > 0
df_with_price_per_mile = df_silver.filter(
    (col("trip_distance") > 0) & 
    (col("fare_amount") > 0)
).withColumn(
    "price_per_mile",
    col("fare_amount") / col("trip_distance")
)

print(f"Records with valid price per mile: {df_with_price_per_mile.count():,}")

# Show sample
display(df_with_price_per_mile.select(
    "pickup_datetime", "pickup_hour", "PULocationID", 
    "trip_distance", "fare_amount", "price_per_mile"
).orderBy("pickup_datetime").limit(10))

In [0]:
# Create aggregated gold table grouped by location and hour
df_gold = df_with_price_per_mile.groupBy("PULocationID", "pickup_hour").agg(
    count("*").alias("trip_count"),
    avg("price_per_mile").alias("avg_fare_per_mile"),
    avg("trip_duration_minutes").alias("avg_trip_duration"),
    avg("trip_distance").alias("avg_trip_distance")
).filter(
    col("trip_count") >= 30
).orderBy("PULocationID", "pickup_hour")

# Cast PULocationID to string
df_gold = df_gold.withColumn("PULocationID", col("PULocationID").cast("string"))

# Write to gold table
df_gold.write.format("delta").mode("overwrite").saveAsTable("workspace.taxi_schema.taxi_gold")

print(f"✓ Created taxi_gold table with {df_gold.count():,} records")
print(f"\nSample records:")
display(df_gold.limit(20))